In [23]:
import numpy as np
import pandas as pd

from pathlib import Path

DATA_DIR = Path("data/raw")

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
products: (32951, 9)
payments: (103886, 5)
reviews: (99224, 7)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


In [7]:
dataset_summary = []

for name, df in datasets.items():
    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)
    })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)

,Dataset,Rows,Columns,Memory (MB)
0,customers,99441,5,29.62
1,orders,99441,8,58.97
2,order_items,112650,7,39.43
3,products,32951,9,6.79
4,payments,103886,5,17.81
5,reviews,99224,7,42.75
6,sellers,3095,4,0.66
7,geolocation,1000163,5,145.20
8,category_translation,71,2,0.01


In [10]:
for name, df in datasets.items():
    print(f"\n {name.upper()} ")

    print("Columns:")
    print(df.columns.tolist())


 CUSTOMERS 
Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

 ORDERS 
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

 ORDER_ITEMS 
Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

 PRODUCTS 
Columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

 PAYMENTS 
Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

 REVIEWS 
Columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

 SELLERS 
Columns:
['seller_id', 'seller_z

In [11]:
for name, df in datasets.items():
    print(f"\n========== {name.upper()} ==========")

    print(df.dtypes)


========== CUSTOMERS ==========
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

========== ORDERS ==========
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

========== ORDER_ITEMS ==========
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

========== PRODUCTS ==========
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    f

In [13]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        missing_summary.append({
            "Dataset" : name,
            "Columns" : column,
            "Missing Values" : missing_count,
            "Missing (%)" : round(missing_count / len(df) * 100, 2)
        })

missing_df = pd.DataFrame(missing_summary)

missing_df = missing_df[
    missing_df["Missing Values"] > 0
]

missing_df = missing_df.sort_values(
    by="Missing (%)",
    ascending = False
)

display(missing_df)


,Dataset,Columns,Missing Values,Missing (%)
37,reviews,review_comment_title,87656,88.34
38,reviews,review_comment_message,58247,58.70
11,orders,order_delivered_customer_date,2965,2.98
22,products,product_name_lenght,610,1.85
21,products,product_category_name,610,1.85
23,products,product_description_lenght,610,1.85
24,products,product_photos_qty,610,1.85
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
26,products,product_length_cm,2,0.01


In [14]:
for name, df in datasets.items():
    duplicated_count = df.duplicated().sum()

    print(f"{name}: {duplicated_count}")

customers: 0
orders: 0
order_items: 0
products: 0
payments: 0
reviews: 0
sellers: 0
geolocation: 261831
category_translation: 0


In [20]:
key_candidates = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

for table_name, columns in key_candidates.items():
    df = datasets[table_name]
    duplicate_count = df.duplicated(subset = columns).sum()
    null_count = df[columns].isnull().sum().sum()

    print(f"\n Table: {table_name}")
    print(f"Key columns: {columns}")
    print(f"Duplicate keys : {duplicate_count}")
    print(f"Null key values: {null_count}")


 Table: customers
Key columns: ['customer_id']
Duplicate keys : 0
Null key values: 0

 Table: orders
Key columns: ['order_id']
Duplicate keys : 0
Null key values: 0

 Table: order_items
Key columns: ['order_id', 'order_item_id']
Duplicate keys : 0
Null key values: 0

 Table: products
Key columns: ['product_id']
Duplicate keys : 0
Null key values: 0

 Table: payments
Key columns: ['order_id', 'payment_sequential']
Duplicate keys : 0
Null key values: 0

 Table: reviews
Key columns: ['review_id']
Duplicate keys : 814
Null key values: 0

 Table: sellers
Key columns: ['seller_id']
Duplicate keys : 0
Null key values: 0

 Table: category_translation
Key columns: ['product_category_name']
Duplicate keys : 0
Null key values: 0


In [17]:
invalid_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Invalid customer references:", len(invalid_customers))

print(
    "Orders without customers:",
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

print(
    "Order items without orders:",
    (~order_items["order_id"].isin(orders["order_id"])).sum()
)

print(
    "Order items without products:",
    (~order_items["product_id"].isin(products["product_id"])).sum()
)

print(
    "Order items without sellers:",
    (~order_items["seller_id"].isin(sellers["seller_id"])).sum()
)

print(
    "Payments without orders:",
    (~payments["order_id"].isin(orders["order_id"])).sum()
)

Invalid customer references: 0
Orders without customers: 0
Order items without orders: 0
Order items without products: 0
Order items without sellers: 0
Payments without orders: 0


In [22]:
quality_report = []

for name, df in datasets.items():

    quality_report.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Total Columns": len(df.columns),
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Total Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

,Dataset,Total Rows,Total Columns,Missing Values,Duplicate Rows,Total Memory (MB)
0,customers,99441,5,0,0,29.62
1,orders,99441,8,4908,0,58.97
2,order_items,112650,7,0,0,39.43
3,products,32951,9,2448,0,6.79
4,payments,103886,5,0,0,17.81
5,reviews,99224,7,145903,0,42.75
6,sellers,3095,4,0,0,0.66
7,geolocation,1000163,5,0,261831,146.09
8,category_translation,71,2,0,0,0.01
